# 🍕 Food Recognition — 95%+ Accuracy
## EfficientNet-B5 | No Kaggle needed | Ready to run!

### ✅ How to run:
1. **Runtime → Change runtime type → GPU (T4)**
2. Run all cells top to bottom — nothing else needed!

## 1️⃣ Check GPU

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name:  {torch.cuda.get_device_name(0)}')
    print(f'GPU VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    print('✅ GPU is ready!')
else:
    raise RuntimeError('❌ GPU not enabled! Runtime → Change runtime type → GPU')

## 2️⃣ Install Packages

In [ ]:
!pip install albumentations timm -q
print('✅ Packages installed!')

## 3️⃣ Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Separate folder from B4 — B4 checkpoints are incompatible with B5!
CHECKPOINT_DIR  = '/content/drive/MyDrive/FoodRecognition_B5'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_LAST = os.path.join(CHECKPOINT_DIR, 'checkpoint_last.pth')
CHECKPOINT_BEST = os.path.join(CHECKPOINT_DIR, 'best_model.pth')

print(f'✅ Drive mounted!')
print(f'   Checkpoints → {CHECKPOINT_DIR}')

## 4️⃣ Download Food101 (No Kaggle needed!)
~5 GB download, takes 3-5 minutes.

In [ ]:
import torchvision
from pathlib import Path

DATA_ROOT = '/content/food101_data'
os.makedirs(DATA_ROOT, exist_ok=True)

print('📥 Downloading Food101... (3-5 minutes, ~5 GB)')
tv_train = torchvision.datasets.Food101(root=DATA_ROOT, split='train', download=True)
tv_test  = torchvision.datasets.Food101(root=DATA_ROOT, split='test',  download=True)

print(f'\n✅ Download complete!')
print(f'   Train: {len(tv_train):,} | Test: {len(tv_test):,} | Classes: {len(tv_train.classes)}')

## 5️⃣ Create Train / Validation / Test Splits

In [ ]:
import torch
import numpy as np
from torch.utils.data import random_split

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = True

total_train = len(tv_train)
val_size    = int(0.20 * total_train)
train_size  = total_train - val_size

train_indices, val_indices = random_split(
    range(total_train), [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
train_indices = list(train_indices)
val_indices   = list(val_indices)

print(f'✅ Train: {train_size:,} | Val: {val_size:,} | Test: {len(tv_test):,}')

## 6️⃣ Imports & Hyperparameters (EfficientNet-B5)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm
import warnings, os
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Hyperparameters ──────────────────────────────────────────
IMAGE_SIZE      = 456    # B5 native resolution (B4 was 380)
BATCH_SIZE      = 12     # Reduce to 8 if CUDA out-of-memory
NUM_WORKERS     = 2
STAGE1_EPOCHS   = 5
STAGE2_EPOCHS   = 40
LR_HEAD         = 3e-4
LR_BACKBONE_TOP = 5e-5
LR_BACKBONE_BOT = 5e-7
WEIGHT_DECAY    = 1e-4
LABEL_SMOOTHING = 0.1
GRAD_CLIP       = 1.0
MIXUP_ALPHA     = 0.4
CUTMIX_ALPHA    = 1.0
CUTMIX_PROB     = 0.5

print(f'✅ Device: {device}')
print(f'   Model: EfficientNet-B5 | Size: {IMAGE_SIZE}px | Batch: {BATCH_SIZE}')

## 7️⃣ Transforms (compatible with albumentations >= 1.4)

In [ ]:
# albumentations >= 1.4 needs size as tuple (H, W)
S     = (IMAGE_SIZE, IMAGE_SIZE)
S_BIG = (int(IMAGE_SIZE * 1.14), int(IMAGE_SIZE * 1.14))

train_transform = A.Compose([
    A.RandomResizedCrop(size=S, scale=(0.6, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.15, p=0.6),
    A.OneOf([A.GaussianBlur(blur_limit=3, p=1.0), A.Sharpen(p=1.0)], p=0.3),
    A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(8, 46), hole_width_range=(8, 46), p=0.4),
    A.Rotate(limit=20, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(height=S_BIG[0], width=S_BIG[1]),
    A.CenterCrop(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

print('✅ Transforms ready (456px, albumentations >= 1.4 compatible)!')

## 8️⃣ Dataset Classes & DataLoaders

In [ ]:
class Food101Dataset(Dataset):
    def __init__(self, tv_dataset, indices, transform=None):
        self.tv_dataset = tv_dataset
        self.indices    = indices
        self.transform  = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        pil_img, label = self.tv_dataset[self.indices[idx]]
        img_np = np.array(pil_img.convert('RGB'))
        if self.transform: img_np = self.transform(image=img_np)['image']
        return img_np, label

class Food101TestDataset(Dataset):
    def __init__(self, tv_dataset, transform=None):
        self.tv_dataset = tv_dataset
        self.transform  = transform
    def __len__(self): return len(self.tv_dataset)
    def __getitem__(self, idx):
        pil_img, label = self.tv_dataset[idx]
        img_np = np.array(pil_img.convert('RGB'))
        if self.transform: img_np = self.transform(image=img_np)['image']
        return img_np, label

train_dataset = Food101Dataset(tv_train, train_indices, transform=train_transform)
val_dataset   = Food101Dataset(tv_train, val_indices,   transform=val_transform)
test_dataset  = Food101TestDataset(tv_test,              transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'✅ DataLoaders ready!')
print(f'   Train: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}')

## 9️⃣ Build Model: EfficientNet-B5 (Noisy-Student)

In [ ]:
class FoodClassifier(nn.Module):
    def __init__(self, num_classes=101):
        super().__init__()
        self.backbone = timm.create_model(
            'tf_efficientnet_b5.ns_jft_in1k',
            pretrained=True, num_classes=0, global_pool='avg'
        )
        self.head = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(self.backbone.num_features, num_classes)
        )
    def forward(self, x):
        return self.head(self.backbone(x))

food_model = FoodClassifier(num_classes=101).to(device)
total = sum(p.numel() for p in food_model.parameters())
print(f'✅ EfficientNet-B5 (Noisy-Student) loaded!')
print(f'   Total params: {total:,} | Features: {food_model.backbone.num_features} | Input: {IMAGE_SIZE}px')

## 🔟 Training Functions (CutMix + Mixup + AMP + Grad Clipping)

In [ ]:
from torch.cuda.amp import autocast, GradScaler

def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    return (np.clip(cx - cut_w//2, 0, W), np.clip(cy - cut_h//2, 0, H),
            np.clip(cx + cut_w//2, 0, W), np.clip(cy + cut_h//2, 0, H))

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    x1, y1, x2, y2 = rand_bbox(x.size(), lam)
    mixed = x.clone(); mixed[:, :, x1:x2, y1:y2] = x[idx, :, x1:x2, y1:y2]
    lam = 1 - (x2-x1)*(y2-y1) / (x.size(2)*x.size(3))
    return mixed, y, y[idx], lam

def mixed_loss(crit, pred, ya, yb, lam):
    return lam * crit(pred, ya) + (1 - lam) * crit(pred, yb)

def train_epoch(model, loader, criterion, optimizer, scaler,
                use_aug=True, mixup_alpha=0.4, cutmix_alpha=1.0, cutmix_prob=0.5):
    model.train()
    total_loss, correct, total = 0.0, 0.0, 0
    pbar = tqdm(loader, desc='  Train', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        if use_aug:
            if np.random.rand() < cutmix_prob:
                images, ya, yb, lam = cutmix_data(images, labels, cutmix_alpha)
            else:
                images, ya, yb, lam = mixup_data(images, labels, mixup_alpha)
        with autocast():
            out = model(images)
            loss = mixed_loss(criterion, out, ya, yb, lam) if use_aug else criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
        _, pred = out.max(1)
        total += labels.size(0)
        correct += (lam*pred.eq(ya).float().sum() + (1-lam)*pred.eq(yb).float().sum()).item() if use_aug \
                   else pred.eq(labels).sum().item()
        pbar.set_postfix({'loss': f'{total_loss/(pbar.n+1):.3f}', 'acc': f'{100.*correct/total:.1f}%'})
    return total_loss/len(loader), 100.*correct/total

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc='  Val  ', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        with autocast():
            out = model(images)
            loss = criterion(out, labels)
        total_loss += loss.item()
        _, pred = out.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
        pbar.set_postfix({'acc': f'{100.*correct/total:.1f}%'})
    return total_loss/len(loader), 100.*correct/total

print('✅ Training functions ready!')

## 1️⃣1️⃣ LLRD Optimizer

In [ ]:
def build_optimizer_llrd(model, head_lr, backbone_top_lr, backbone_bot_lr, weight_decay):
    groups = []
    try:
        blocks = list(model.backbone.blocks)
        n = len(blocks)
        for i, block in enumerate(blocks):
            lr = backbone_bot_lr + (backbone_top_lr - backbone_bot_lr) * (i / max(n-1, 1))
            groups.append({'params': list(block.parameters()), 'lr': lr})
        groups.append({'params': list(model.backbone.conv_stem.parameters()), 'lr': backbone_bot_lr})
        groups.append({'params': list(model.backbone.bn1.parameters()),       'lr': backbone_bot_lr})
        for attr in ['conv_head', 'bn2']:
            if hasattr(model.backbone, attr):
                groups.append({'params': list(getattr(model.backbone, attr).parameters()), 'lr': backbone_top_lr})
    except Exception as e:
        print(f'   LLRD fallback: {e}')
        groups = [{'params': list(model.backbone.parameters()), 'lr': backbone_top_lr}]
    groups.append({'params': list(model.head.parameters()), 'lr': head_lr})
    opt = optim.AdamW(groups, weight_decay=weight_decay)
    print(f'   Built optimizer with {len(groups)} param groups')
    return opt

print('✅ LLRD optimizer function ready!')

## 1️⃣2️⃣ Stage 1 — Head Warmup (backbone frozen)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scaler    = GradScaler()

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc  = 0.0
current_stage = 1
start_epoch   = 0

if os.path.exists(CHECKPOINT_LAST):
    print('🔄 Checkpoint found — resuming...')
    ckpt = torch.load(CHECKPOINT_LAST, map_location=device)
    food_model.load_state_dict(ckpt['model_state_dict'])
    history       = ckpt.get('history', history)
    best_val_acc  = ckpt.get('best_val_acc', 0.0)
    current_stage = ckpt.get('stage', 1)
    start_epoch   = ckpt.get('epoch', 0) + 1
    print(f'   Stage {current_stage}, epoch {start_epoch}, best {best_val_acc:.2f}%')
else:
    print('🆕 Starting fresh training')

if current_stage == 1:
    print(f'\n' + '='*65)
    print(f'📌 STAGE 1: Head-only warmup ({STAGE1_EPOCHS} epochs)')
    print('='*65)

    for p in food_model.backbone.parameters(): p.requires_grad = False
    for p in food_model.head.parameters():     p.requires_grad = True

    opt_s1 = optim.AdamW(food_model.head.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
    sch_s1 = optim.lr_scheduler.OneCycleLR(
        opt_s1, max_lr=LR_HEAD, steps_per_epoch=len(train_loader),
        epochs=STAGE1_EPOCHS, pct_start=0.3)

    for epoch in range(start_epoch, STAGE1_EPOCHS):
        print(f'\n  Epoch {epoch+1}/{STAGE1_EPOCHS}')
        tl, ta = train_epoch(food_model, train_loader, criterion, opt_s1, scaler, use_aug=False)
        vl, va = validate(food_model, val_loader, criterion)
        sch_s1.step()
        history['train_loss'].append(tl); history['train_acc'].append(ta)
        history['val_loss'].append(vl);   history['val_acc'].append(va)
        print(f'  → Train: {ta:.2f}%  |  Val: {va:.2f}%')
        torch.save({'epoch': epoch, 'model_state_dict': food_model.state_dict(),
                    'history': history, 'best_val_acc': best_val_acc, 'stage': 1}, CHECKPOINT_LAST)
        if va > best_val_acc:
            best_val_acc = va
            torch.save({'epoch': epoch, 'model_state_dict': food_model.state_dict(), 'val_acc': va}, CHECKPOINT_BEST)
            print(f'  ✅ Best saved! ({va:.2f}%)')

    print(f'\n✅ Stage 1 complete! Best val: {best_val_acc:.2f}%')
    current_stage = 2
    start_epoch   = 0

print('\n→ Moving to Stage 2...')

## 1️⃣3️⃣ Stage 2 — Full Fine-Tuning with LLRD

In [ ]:
print('🔓 Unfreezing all layers...')
for p in food_model.parameters(): p.requires_grad = True
print(f'   Trainable: {sum(p.numel() for p in food_model.parameters() if p.requires_grad):,}')

opt_s2 = build_optimizer_llrd(food_model, LR_HEAD, LR_BACKBONE_TOP, LR_BACKBONE_BOT, WEIGHT_DECAY)
sch_s2 = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt_s2, T_0=10, T_mult=2, eta_min=1e-8)

if os.path.exists(CHECKPOINT_LAST):
    ckpt = torch.load(CHECKPOINT_LAST, map_location=device)
    if ckpt.get('stage', 1) == 2:
        food_model.load_state_dict(ckpt['model_state_dict'])
        start_epoch  = ckpt['epoch'] + 1
        history      = ckpt.get('history', history)
        best_val_acc = ckpt.get('best_val_acc', best_val_acc)
        try: opt_s2.load_state_dict(ckpt['optimizer_state_dict'])
        except: pass
        try: sch_s2.load_state_dict(ckpt['scheduler_state_dict'])
        except: pass
        print(f'   Resumed stage 2 from epoch {start_epoch}, best {best_val_acc:.2f}%')

print(f'\n' + '='*65)
print(f'🚀 STAGE 2: Full fine-tuning ({STAGE2_EPOCHS} epochs) — Target: 95%+')
print('='*65)

for epoch in range(start_epoch, STAGE2_EPOCHS):
    print(f'\n📍 Epoch {epoch+1}/{STAGE2_EPOCHS}')
    print('-'*65)

    use_aug = epoch < STAGE2_EPOCHS - 5
    if epoch == STAGE2_EPOCHS - 5:
        print('  ⚠️  Disabling CutMix/Mixup for final convergence epochs')

    tl, ta = train_epoch(food_model, train_loader, criterion, opt_s2, scaler,
                         use_aug=use_aug, mixup_alpha=MIXUP_ALPHA,
                         cutmix_alpha=CUTMIX_ALPHA, cutmix_prob=CUTMIX_PROB)
    vl, va = validate(food_model, val_loader, criterion)
    sch_s2.step()

    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)

    print(f'  Train: {ta:.2f}%  |  Val: {va:.2f}%  |  Gap to 95%: {max(95.-va,0):.2f}%  |  Best: {best_val_acc:.2f}%')
    if ta - va > 10: print(f'  ⚠️  Possible overfitting (gap: {ta-va:.1f}%)')

    torch.save({'epoch': epoch, 'model_state_dict': food_model.state_dict(),
                'optimizer_state_dict': opt_s2.state_dict(),
                'scheduler_state_dict': sch_s2.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'history': history, 'best_val_acc': best_val_acc, 'stage': 2}, CHECKPOINT_LAST)

    if va > best_val_acc:
        best_val_acc = va
        torch.save({'epoch': epoch, 'model_state_dict': food_model.state_dict(), 'val_acc': va}, CHECKPOINT_BEST)
        print(f'  ✅ NEW BEST: {va:.2f}%!')

print('\n' + '='*65)
print(f'🎉 Training complete! Best val: {best_val_acc:.2f}%')
if best_val_acc >= 95: print('🎉🎉🎉 TARGET REACHED: 95%+ ACHIEVED!')
elif best_val_acc >= 90: print('📈 Very close! Increase STAGE2_EPOCHS and re-run this cell.')
print('='*65)

## 1️⃣4️⃣ Final Evaluation: Standard + TTA (8 crops)

In [ ]:
print('📥 Loading best model...')
ckpt = torch.load(CHECKPOINT_BEST, map_location=device)
food_model.load_state_dict(ckpt['model_state_dict'])
food_model.eval()
print(f'   Epoch {ckpt["epoch"]+1}, val acc {ckpt["val_acc"]:.2f}%')

print('\n🔍 Standard test evaluation...')
test_loss, test_acc = validate(food_model, test_loader, criterion)
print(f'   📊 Standard Test Accuracy: {test_acc:.2f}%')

In [ ]:
@torch.no_grad()
def evaluate_tta(model, loader, num_tta=8):
    model.eval()
    correct, total = 0, 0
    IS = IMAGE_SIZE
    tta_tf = [
        A.Compose([A.Resize(int(IS*1.14), int(IS*1.14)), A.CenterCrop(height=IS, width=IS),
                   A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(int(IS*1.14), int(IS*1.14)), A.CenterCrop(height=IS, width=IS),
                   A.HorizontalFlip(p=1), A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(int(IS*1.25), int(IS*1.25)), A.CenterCrop(height=IS, width=IS),
                   A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(int(IS*1.25), int(IS*1.25)), A.CenterCrop(height=IS, width=IS),
                   A.HorizontalFlip(p=1), A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(int(IS*1.05), int(IS*1.05)), A.CenterCrop(height=IS, width=IS),
                   A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(int(IS*1.05), int(IS*1.05)), A.CenterCrop(height=IS, width=IS),
                   A.HorizontalFlip(p=1), A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IS, IS), A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2()]),
        A.Compose([A.Resize(IS, IS), A.HorizontalFlip(p=1),
                   A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2()]),
    ][:num_tta]

    pbar = tqdm(loader, desc=f'  TTA ({num_tta} augs)')
    for images, labels in pbar:
        labels = labels.to(device)
        preds  = []
        for t in tta_tf:
            aug = []
            for img in images:
                np_img = img.permute(1,2,0).numpy()
                np_img = (np_img * [0.229,0.224,0.225] + [0.485,0.456,0.406]) * 255
                np_img = np_img.clip(0,255).astype(np.uint8)
                aug.append(t(image=np_img)['image'])
            with autocast(): out = model(torch.stack(aug).to(device))
            preds.append(torch.softmax(out, dim=1))
        _, predicted = torch.stack(preds).mean(0).max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        pbar.set_postfix({'acc': f'{100.*correct/total:.2f}%'})
    return 100.*correct/total

print('🔍 Running TTA evaluation (8 augmentations per image)...\n')
tta_acc = evaluate_tta(food_model, test_loader, num_tta=8)

print(f'\n' + '='*65)
print(f'📊 FINAL RESULTS:')
print(f'   Standard Test Accuracy:      {test_acc:.2f}%')
print(f'   TTA Test Accuracy (8 crops):  {tta_acc:.2f}%')
print(f'   TTA Boost:                   +{tta_acc - test_acc:.2f}%')
print('='*65)
if tta_acc >= 95:    print('🎉🎉🎉  95%+ ACHIEVED!')
elif test_acc >= 93: print('📈 Very close! Increase STAGE2_EPOCHS and re-run Stage 2.')

## 1️⃣5️⃣ Plot Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history['train_acc']) + 1)
ax1.plot(ep, history['train_acc'], label='Train', color='steelblue', lw=2)
ax1.plot(ep, history['val_acc'],   label='Val',   color='darkorange', lw=2)
ax1.axhline(95,           color='red',   ls='--', label='95% Target', lw=1.5)
ax1.axhline(best_val_acc, color='green', ls=':',  label=f'Best {best_val_acc:.1f}%', lw=1.5)
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('%')
ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(ep, history['train_loss'], label='Train', color='steelblue', lw=2)
ax2.plot(ep, history['val_loss'],   label='Val',   color='darkorange', lw=2)
ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.suptitle(f'EfficientNet-B5  |  Best Val: {best_val_acc:.2f}%  |  Test: {test_acc:.2f}%  |  TTA: {tta_acc:.2f}%')
plt.tight_layout()
plt.savefig('training_history_b5.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved!')

## 1️⃣6️⃣ Download Best Model

In [ ]:
from google.colab import files
import shutil

shutil.copy(CHECKPOINT_BEST, 'best_model_efficientnet_b5.pth')
files.download('best_model_efficientnet_b5.pth')
files.download('training_history_b5.png')

print(f'✅ Downloaded!')
print(f'   Best val:        {best_val_acc:.2f}%')
print(f'   Test (standard): {test_acc:.2f}%')
print(f'   Test (TTA 8x):   {tta_acc:.2f}%')